<a href="https://colab.research.google.com/github/Swag-Pseudopy/Conic-Particle-Methods-for-MNE/blob/main/CPMP_Slingshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# cpmp_sling_fixed.py
# KEY FIXES:
# 1. Position step uses PRE-UPDATE weights (saved before weight update)
# 2. Adaptive clipping: max step scales with weight
# 3. Mirror Prox correction step uses |alpha|, |beta| (magnitude only) for the
#    correction gradient application — the slingshot sign lives only in the
#    prediction step, not the correction step
# 4. Added entropy + duality gap to history and 4-panel dashboard

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# ==========================================
# 1. Slingshot Stepsize Scheduler
# ==========================================
def get_slingshot_steps(problem_type, t, T, L=None, m=None, M=None, rng=None):
    rng = rng or np.random.default_rng()
    if problem_type == "bilinear":
        k = t // 2
        r_t = (M + m) / 2 + (M - m) / 2 * np.cos((2 * k + 1) * np.pi / (2 * T))
        h = 1.0 / np.sqrt(r_t)
        return (h, -h) if t % 2 == 0 else (-h, h)
    else:
        h = 1.0 / (3.0 * L)
        if t % 2 == 0:
            return (h, -h) if rng.random() < 0.5 else (-h, h)
        else:
            return h, h

# ==========================================
# 2. Conic Particle Mirror Prox (CP-MP) — Fixed
# ==========================================
class ConicParticlesMP:
    def __init__(self, n, m, d, rng=None):
        rng = rng or np.random.default_rng()
        self.a = np.ones(n) / n
        self.x = rng.uniform(-2, 2, (n, d))
        self.b = np.ones(m) / m
        self.y = rng.uniform(-2, 2, (m, d))

    def _step_weights_only(self, a, b, grad_a, grad_b, alpha, beta):
        """Returns new weights only (log-sum-exp stable)."""
        log_a = np.log(np.maximum(a, 1e-15)) - alpha * grad_a
        log_a -= np.max(log_a)
        a_new = np.exp(log_a); a_new /= a_new.sum()

        log_b = np.log(np.maximum(b, 1e-15)) + beta * grad_b
        log_b -= np.max(log_b)
        b_new = np.exp(log_b); b_new /= b_new.sum()
        return a_new, b_new

    def _step_positions_only(self, a_ref, x, b_ref, y, grad_x, grad_y, alpha, beta):
        """
        Position step using REFERENCE weights (pre-update) for preconditioning.
        Adaptive clip: smaller for heavy particles.
        """
        denom_x = np.maximum(a_ref[:, None], 1e-3)
        step_x = alpha * grad_x / denom_x
        clip_x = np.minimum(1.0, 0.1 / denom_x)
        x_new = x - np.clip(step_x, -clip_x, clip_x)

        denom_y = np.maximum(b_ref[:, None], 1e-3)
        step_y = beta * grad_y / denom_y
        clip_y = np.minimum(1.0, 0.1 / denom_y)
        y_new = y + np.clip(step_y, -clip_y, clip_y)

        x_new = np.clip(x_new, -2.5, 2.5)
        y_new = np.clip(y_new, -2.5, 2.5)
        return x_new, y_new

    def _full_step(self, a, x, b, y, grad_a, grad_x, grad_b, grad_y, alpha, beta):
        """Full step: weights then positions (positions use pre-weight-update ref)."""
        a_ref = a.copy(); b_ref = b.copy()
        a_new, b_new = self._step_weights_only(a, b, grad_a, grad_b, alpha, beta)
        x_new, y_new = self._step_positions_only(a_ref, x, b_ref, y, grad_x, grad_y, alpha, beta)
        return a_new, x_new, b_new, y_new

    def update_mp(self, calc_grads_fn, alpha, beta):
        """
        Mirror Prox:
          Prediction: full step with (alpha, beta) to get predicted state
          Correction:  apply correction gradients from predicted state,
                       but use |alpha|, |beta| for the final step magnitude.
                       The slingshot SIGN was for the prediction; the correction
                       should always be a genuine descent/ascent step.
        """
        # STEP 1: PREDICTION — gradients at current, step with slingshot signs
        ga1, gx1, gb1, gy1 = calc_grads_fn(self.a, self.x, self.b, self.y)
        a_p, x_p, b_p, y_p = self._full_step(
            self.a, self.x, self.b, self.y, ga1, gx1, gb1, gy1, alpha, beta)

        # STEP 2: CORRECTION — gradients at predicted state
        ga2, gx2, gb2, gy2 = calc_grads_fn(a_p, x_p, b_p, y_p)

        # Apply correction from ORIGINAL state using |alpha|, |beta|
        # (the sign of the slingshot already shaped the prediction step;
        #  the correction is a genuine improvement step)
        self.a, self.x, self.b, self.y = self._full_step(
            self.a, self.x, self.b, self.y,
            ga2, gx2, gb2, gy2,
            abs(alpha), abs(beta)   # <-- KEY FIX: correction always positive
        )

# ==========================================
# 3. Objective Functions & Gradients
# ==========================================
def f_bilinear(x, y, B):    return float(x @ B @ y)
def grad_bilinear(x, y, B): return B @ y, B.T @ x

def logcosh(x):
    s = np.sign(x) * x
    return s + np.log1p(np.exp(-2*s)) - np.log(2.0)

def f_convex_concave(x, y):    return float(logcosh(x) - logcosh(y))
def grad_convex_concave(x, y): return np.tanh(x), -np.tanh(y)

def f_sc_sc(x, y, mu=1.0):    return float(0.5*mu*np.sum(x**2) - 0.5*mu*np.sum(y**2) + np.sum(x*y))
def grad_sc_sc(x, y, mu=1.0): return mu*x + y, -mu*y + x

# ==========================================
# 4. Diagnostics helpers
# ==========================================
def get_weighted_barycenter(weights, positions):
    return np.sum(weights[:, None] * positions, axis=0)

def entropy(weights):
    w = np.maximum(weights, 1e-15)
    return float(-np.sum(w * np.log(w)))

def duality_gap(a, x, b, y, f_func):
    vals = np.array([[float(f_func(x[i], y[j])) for j in range(len(y))] for i in range(len(x))])
    return float(np.max(vals.T @ a) - np.min(vals @ b))

# ==========================================
# 5. Experiment runner
# ==========================================
def run_experiment_mp(model, grad_func, f_func, steps=200,
                      problem_type="bilinear", T=None, L=None, m=None, M=None,
                      rng=None):
    rng = rng or np.random.default_rng()
    T = T or steps
    history = {'a': [], 'x': [], 'b': [], 'y': [],
               'barycenter': [], 'entropy_a': [], 'entropy_b': [],
               'hamiltonian': [], 'gap': []}

    for t in range(steps):
        history['a'].append(model.a.copy())
        history['x'].append(model.x.copy())
        history['b'].append(model.b.copy())
        history['y'].append(model.y.copy())

        xb = get_weighted_barycenter(model.a, model.x)[0]
        yb = get_weighted_barycenter(model.b, model.y)[0]
        history['barycenter'].append((xb, yb))
        history['entropy_a'].append(entropy(model.a))
        history['entropy_b'].append(entropy(model.b))

        gx0, gy0 = grad_func(np.array([xb]), np.array([yb]))
        history['hamiltonian'].append(0.5*(float(np.sum(gx0**2)) + float(np.sum(gy0**2))))
        history['gap'].append(duality_gap(model.a, model.x, model.b, model.y, f_func) if t % 10 == 0 else np.nan)

        alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M, rng=rng)

        def compute_grads(a_s, x_s, b_s, y_s):
            n_, m_ = len(x_s), len(y_s)
            gx_t = np.zeros_like(x_s); gy_t = np.zeros_like(y_s)
            ga_t = np.zeros(n_);        gb_t = np.zeros(m_)
            for i in range(n_):
                for j in range(m_):
                    gx, gy = grad_func(x_s[i], y_s[j])
                    val = float(f_func(x_s[i], y_s[j]))
                    gx_t[i] += b_s[j] * float(gx)
                    gy_t[j] += a_s[i] * float(gy)
                    ga_t[i] += b_s[j] * val
                    gb_t[j] += a_s[i] * val
            return ga_t, gx_t, gb_t, gy_t

        model.update_mp(compute_grads, alpha, beta)

    return history

# ==========================================
# 6. 4-panel dashboard
# ==========================================
def plot_dashboard(history, grad_func, title=""):
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    xs = np.linspace(-3, 3, 80)
    X, Y = np.meshgrid(xs, xs)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i,j]]), np.array([Y[i,j]]))
            Z[i,j] = 0.5*(float(np.sum(gx**2)) + float(np.sum(gy**2)))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.5)

    traj = np.array(history['barycenter'])
    mask = ~np.isnan(traj).any(axis=1)
    tc = traj[mask]
    if len(tc):
        axs[0].plot(tc[:,0], tc[:,1], 'k-', lw=2)
        axs[0].scatter(*tc[0], c='white', edgecolors='k', s=100, zorder=4)
        axs[0].scatter(*tc[-1], c='yellow', edgecolors='k', marker='*', s=400, zorder=5)
    a_f, x_f = history['a'][-1], history['x'][-1]
    b_f, y_f = history['b'][-1], history['y'][-1]
    axs[0].scatter(x_f[:,0], np.full(len(x_f), -0.05),
                   s=np.maximum(a_f*2000, 5), c='red', alpha=0.8, edgecolors='k')
    axs[0].scatter(np.full(len(y_f), 0.05), y_f[:,0],
                   s=np.maximum(b_f*2000, 5), c='orange', alpha=0.8, edgecolors='k')
    axs[0].set_xlim(-3,3); axs[0].set_ylim(-3,3); axs[0].set_title("Hamiltonian + Particles")

    if len(tc):
        dist = np.sqrt(tc[:,0]**2 + tc[:,1]**2)
        axs[1].semilogy(dist, c='purple', lw=2)
        axs[1].set_title("Barycenter Dist (log)")
        axs[1].set_xlabel("Iteration"); axs[1].grid(True, ls='--', alpha=0.5)

    ea = np.array(history['entropy_a']); eb = np.array(history['entropy_b'])
    axs[2].plot(ea, 'r-', lw=2, label='H(a)')
    axs[2].plot(eb, color='orange', lw=2, label='H(b)')
    axs[2].axhline(0, color='k', ls='--', lw=1)
    axs[2].set_title("Strategy Entropy (↓ → collapse)")
    axs[2].set_xlabel("Iteration"); axs[2].legend(fontsize=8); axs[2].grid(True, ls='--', alpha=0.5)

    gap = np.array(history['gap'])
    vi = ~np.isnan(gap)
    axs[3].semilogy(np.where(vi)[0], gap[vi], 'g-o', lw=2, ms=4)
    axs[3].axhline(0.01, color='red', ls='--', lw=1, label='ε=0.01')
    axs[3].set_title("Duality Gap (↓ → Nash)")
    axs[3].set_xlabel("Iteration"); axs[3].legend(fontsize=8); axs[3].grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    return fig

# ==========================================
# 7. GIF Generator
# ==========================================
def animate_convergence(history, grad_func, filename="convergence.gif", title=""):
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    xs = np.linspace(-3, 3, 80)
    X, Y = np.meshgrid(xs, xs)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i,j]]), np.array([Y[i,j]]))
            Z[i,j] = 0.5*(float(np.sum(gx**2)) + float(np.sum(gy**2)))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.4)
    scat_p1 = axs[0].scatter([], [], c='red', edgecolors='k', zorder=5)
    scat_p2 = axs[0].scatter([], [], c='orange', edgecolors='k', zorder=4)
    line_bary, = axs[0].plot([], [], 'k-', lw=2, zorder=2)
    scat_bary = axs[0].scatter([], [], c='yellow', edgecolors='k', marker='*', s=300, zorder=10)
    axs[0].set_xlim(-3,3); axs[0].set_ylim(-3,3); axs[0].set_title(title)

    steps_range = np.arange(len(history['a']))
    ea = np.array(history['entropy_a']); eb = np.array(history['entropy_b'])

    def update(frame):
        a, x = history['a'][frame], history['x'][frame]
        b, y = history['b'][frame], history['y'][frame]
        scat_p1.set_offsets(np.c_[x[:,0], np.full(len(x), -0.05)])
        scat_p1.set_sizes(np.maximum(a*1500, 5))
        scat_p2.set_offsets(np.c_[np.full(len(y), 0.05), y[:,0]])
        scat_p2.set_sizes(np.maximum(b*1500, 5))
        bp = np.array(history['barycenter'][:frame+1])
        line_bary.set_data(bp[:,0], bp[:,1])
        scat_bary.set_offsets([bp[-1]])

        axs[1].clear(); axs[1].set_xlim(-3,3); axs[1].set_ylim(0,1.1)
        axs[1].set_title(f"P1 Mass (H={ea[frame]:.2f})")
        axs[1].bar(x[:,0], a, width=0.15, color='red', alpha=0.7, edgecolor='k')

        axs[2].clear(); axs[2].set_xlim(-3,3); axs[2].set_ylim(0,1.1)
        axs[2].set_title(f"P2 Mass (H={eb[frame]:.2f})")
        axs[2].bar(y[:,0], b, width=0.15, color='orange', alpha=0.7, edgecolor='k')

        axs[3].clear()
        axs[3].plot(steps_range[:frame+1], ea[:frame+1], 'r-', lw=1.5, label='H(a)')
        axs[3].plot(steps_range[:frame+1], eb[:frame+1], color='orange', lw=1.5, label='H(b)')
        axs[3].set_xlim(0, len(history['a']))
        axs[3].set_ylim(-0.1, np.log(len(a))+0.2)
        axs[3].set_title("Entropy over time"); axs[3].legend(fontsize=7)
        axs[3].grid(True, ls='--', alpha=0.4)

    print(f"Generating {filename}...")
    ani = animation.FuncAnimation(fig, update, frames=len(history['a']), interval=80)
    ani.save(filename, writer='pillow')
    plt.close(fig)
    print("Done!")

# ==========================================
# 8. Driver
# ==========================================
if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # ---- Case 1: Bilinear ----
    print("=== Bilinear (CP-MP fixed) ===")
    B_mat = np.array([[1.0]])
    model_bl = ConicParticlesMP(n=30, m=30, d=1, rng=rng)
    history_bl = run_experiment_mp(
        model_bl,
        lambda x,y: grad_bilinear(x,y,B_mat),
        lambda x,y: f_bilinear(x,y,B_mat),
        steps=100, problem_type="bilinear", m=1.0, M=1.0, rng=rng
    )
    fig = plot_dashboard(history_bl, lambda x,y: grad_bilinear(x,y,B_mat),
                         title="Bilinear - CP-MP Fixed")
    fig.savefig("cpmp_bilinear_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_bl, lambda x,y: grad_bilinear(x,y,B_mat),
                        filename="cpmp_bilinear.gif", title="Bilinear CP-MP")

    # ---- Case 2: Convex-Concave ----
    print("\n=== Convex-Concave (CP-MP fixed) ===")
    model_cc = ConicParticlesMP(n=30, m=30, d=1, rng=rng)
    history_cc = run_experiment_mp(
        model_cc, grad_convex_concave, f_convex_concave,
        steps=200, problem_type="nonlinear", L=1.0, rng=rng
    )
    fig = plot_dashboard(history_cc, grad_convex_concave,
                         title="Convex-Concave - CP-MP Fixed")
    fig.savefig("cpmp_cc_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_cc, grad_convex_concave,
                        filename="cpmp_cc.gif", title="Conv-Conc CP-MP")

    # ---- Case 3: SC-SC ----
    print("\n=== SC-SC (CP-MP fixed) ===")
    model_sc = ConicParticlesMP(n=30, m=30, d=1, rng=rng)
    history_sc = run_experiment_mp(
        model_sc,
        lambda x,y: grad_sc_sc(x,y,mu=1.0),
        lambda x,y: f_sc_sc(x,y,mu=1.0),
        steps=150, problem_type="nonlinear", L=np.sqrt(2), rng=rng
    )
    fig = plot_dashboard(history_sc, lambda x,y: grad_sc_sc(x,y,mu=1.0),
                         title="SC-SC - CP-MP Fixed")
    fig.savefig("cpmp_scsc_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_sc, lambda x,y: grad_sc_sc(x,y,mu=1.0),
                        filename="cpmp_scsc.gif", title="SC-SC CP-MP")

    print("\nAll done!")

=== Bilinear (CP-MP fixed) ===


/tmp/ipykernel_5960/3669591081.py:173: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gx_t[i] += b_s[j] * float(gx)
/tmp/ipykernel_5960/3669591081.py:174: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gy_t[j] += a_s[i] * float(gy)


Generating cpmp_bilinear.gif...
Done!

=== Convex-Concave (CP-MP fixed) ===


/tmp/ipykernel_5960/3669591081.py:115: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  def f_convex_concave(x, y):    return float(logcosh(x) - logcosh(y))
/tmp/ipykernel_5960/3669591081.py:173: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gx_t[i] += b_s[j] * float(gx)
/tmp/ipykernel_5960/3669591081.py:174: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gy_t[j] += a_s[i] * float(gy)


Generating cpmp_cc.gif...
Done!

=== SC-SC (CP-MP fixed) ===


/tmp/ipykernel_5960/3669591081.py:173: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gx_t[i] += b_s[j] * float(gx)
/tmp/ipykernel_5960/3669591081.py:174: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gy_t[j] += a_s[i] * float(gy)


Generating cpmp_scsc.gif...
Done!

All done!


!pip install pillow

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Slingshot Stepsize Scheduler
# ==========================================
def get_slingshot_steps(problem_type, t, T, L=None, m=None, M=None, rng=None):
    rng = rng or np.random.default_rng()

    if problem_type == "bilinear":
        k = t // 2
        r_t = (M + m) / 2 + (M - m) / 2 * np.cos((2 * k + 1) * np.pi / (2 * T))
        h = 1.0 / np.sqrt(r_t)
        return (h, -h) if t % 2 == 0 else (-h, h)
    else:
        h = 1.0 / (3.0 * L)
        if t % 2 == 0:
            if rng.random() < 0.5:
                return h, -h
            else:
                return -h, h
        else:
            return h, h

# ==========================================
# 2. Conic Particle Mirror Prox (CP-MP)
# ==========================================
class ConicParticlesMP:
    def __init__(self, n, m, d, rng=None):
        rng = rng or np.random.default_rng()
        self.a = np.ones(n) / n
        self.x = rng.uniform(-2, 2, (n, d))
        self.b = np.ones(m) / m
        self.y = rng.uniform(-2, 2, (m, d))

    def _step_from_state(self, a, x, b, y, grad_a, grad_x, grad_b, grad_y, alpha, beta):
        """
        Takes a single state and gradients, and returns the NEW state.
        Includes all numerical stability fixes (Log-Sum-Exp & Physical bounding).
        """
        # ---- Player 1 (Descent) ----
        log_a = np.log(np.maximum(a, 1e-15)) - alpha * grad_a
        log_a -= np.max(log_a)
        a_new = np.exp(log_a)
        a_new /= np.sum(a_new)

        step_x = alpha * grad_x / np.maximum(a[:, None], 1e-3)
        x_new = x - np.clip(step_x, -0.5, 0.5)

        # ---- Player 2 (Ascent) ----
        log_b = np.log(np.maximum(b, 1e-15)) + beta * grad_b
        log_b -= np.max(log_b)
        b_new = np.exp(log_b)
        b_new /= np.sum(b_new)

        step_y = beta * grad_y / np.maximum(b[:, None], 1e-3)
        y_new = y + np.clip(step_y, -0.5, 0.5)

        return a_new, x_new, b_new, y_new

    def update_mp(self, calc_grads_fn, alpha, beta):
        """
        Performs the two-step Mirror Prox (Prediction-Correction) update.
        """
        # STEP 1: PREDICTION (Calculate gradients at current state)
        ga_1, gx_1, gb_1, gy_1 = calc_grads_fn(self.a, self.x, self.b, self.y)

        # Peek into the future to get the predicted state (_p)
        a_p, x_p, b_p, y_p = self._step_from_state(
            self.a, self.x, self.b, self.y,
            ga_1, gx_1, gb_1, gy_1, alpha, beta
        )

        # STEP 2: CORRECTION (Calculate gradients at PREDICTED state)
        ga_2, gx_2, gb_2, gy_2 = calc_grads_fn(a_p, x_p, b_p, y_p)

        # Take the final step from the ORIGINAL state using CORRECTION gradients
        self.a, self.x, self.b, self.y = self._step_from_state(
            self.a, self.x, self.b, self.y,
            ga_2, gx_2, gb_2, gy_2, alpha, beta
        )

# ==========================================
# 3. Objective Functions
# ==========================================
def f_bilinear(x, y, B): return x @ B @ y
def grad_bilinear(x, y, B): return B @ y, B.T @ x

def logcosh(x):
    s = np.sign(x) * x
    p = np.exp(-2 * s)
    return s + np.log1p(p) - np.log(2.0)

def f_convex_concave(x, y): return logcosh(x) - logcosh(y)
def grad_convex_concave(x, y): return np.tanh(x), -np.tanh(y)

def f_sc_sc(x, y, mu=1.0): return 0.5 * mu * np.sum(x**2) - 0.5 * mu * np.sum(y**2) + np.sum(x * y)
def grad_sc_sc(x, y, mu=1.0): return mu * x + y, -mu * y + x

# ==========================================
# 4. Advanced Diagnostics Plotter
# ==========================================
def get_weighted_barycenter(weights, positions):
    return np.sum(weights[:, None] * positions, axis=0)

def plot_advanced_diagnostics(grad_func, traj, model, title=""):
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    xs, ys = np.linspace(-3, 3, 100), np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)

    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (np.sum(gx**2) + np.sum(gy**2))

    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.5, zorder=1)

    traj = np.array(traj)
    valid_mask = ~np.isnan(traj).any(axis=1)
    traj_clean = traj[valid_mask]

    if len(traj_clean) > 0:
        axs[0].plot(traj_clean[:, 0], traj_clean[:, 1], c='black', linestyle='-', linewidth=2, label='Barycenter Path', zorder=2)
        axs[0].scatter(traj_clean[0, 0], traj_clean[0, 1], c='white', edgecolors='black', marker='o', s=100, label='Start', zorder=3)
        axs[0].scatter(traj_clean[-1, 0], traj_clean[-1, 1], c='yellow', edgecolors='black', marker='*', s=400, label='End (Eq)', zorder=10)

    p1_y_offset = np.full_like(model.x[:, 0], -0.1)
    p2_x_offset = np.full_like(model.y[:, 0], 0.1)
    s_p1 = np.maximum(model.a * 1500, 10)
    s_p2 = np.maximum(model.b * 1500, 10)

    axs[0].scatter(model.x[:, 0], p1_y_offset, s=s_p1, c='red', alpha=0.8, edgecolors='black', label='P1 Particles', zorder=5)
    axs[0].scatter(p2_x_offset, model.y[:, 0], s=s_p2, c='orange', alpha=0.8, edgecolors='black', label='P2 Particles', zorder=4)

    axs[0].set_xlim(-3, 3)
    axs[0].set_ylim(-3, 3)
    axs[0].set_title(f"{title} - Hamiltonian Landscape")
    axs[0].legend(loc='upper right', fontsize=8)
    axs[0].grid(True, linestyle='--', alpha=0.5, zorder=0)

    if len(traj_clean) > 0:
        dist_to_origin = np.sqrt(traj_clean[:, 0]**2 + traj_clean[:, 1]**2)
        axs[1].plot(dist_to_origin, label='Barycenter Dist to (0,0)', c='purple', linewidth=2)
        axs[1].set_yscale('log')
        axs[1].set_title("Convergence over Time")
        axs[1].set_xlabel("Iteration")
        axs[1].set_ylabel("Log Distance")
        axs[1].legend()
        axs[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# # ==========================================
# # 5. Mirror Prox Experiment Runner
# # ==========================================
# def run_experiment_mp(model, grad_func, f_func, steps=100, problem_type="bilinear", T=None, L=None, m=None, M=None):
#     trajectory = []
#     T = T or steps

#     for t in range(steps):
#         # Log Current Barycenter
#         x_bar = get_weighted_barycenter(model.a, model.x)[0]
#         y_bar = get_weighted_barycenter(model.b, model.y)[0]
#         trajectory.append((x_bar, y_bar))

#         alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M)

#         # We wrap the gradient computation in a function so update_mp can call it
#         # on both the current state AND the predicted future state.
#         def compute_system_grads(a_s, x_s, b_s, y_s):
#             n, m_ = x_s.shape[0], y_s.shape[0]
#             grad_x, grad_y = np.zeros_like(x_s), np.zeros_like(y_s)
#             grad_a, grad_b = np.zeros(n), np.zeros(m_)

#             for i in range(n):
#                 for j in range(m_):
#                     gx, gy = grad_func(x_s[i], y_s[j])
#                     val = f_func(x_s[i], y_s[j])

#                     grad_x[i] += b_s[j] * float(gx)
#                     grad_y[j] += a_s[i] * float(gy)
#                     grad_a[i] += b_s[j] * float(val)
#                     grad_b[j] += a_s[i] * float(val)
#             return grad_a, grad_x, grad_b, grad_y

#         # Execute the Mirror Prox update
#         model.update_mp(compute_system_grads, alpha, beta)

#     return trajectory

# # ==========================================
# # 6. Execute Scenarios
# # ==========================================
# if __name__ == "__main__":
#     rng = np.random.default_rng(42)

#     print("Running Bilinear CP-MP...")
#     B = np.array([[1.0]])
#     model_bl = ConicParticlesMP(n=20, m=20, d=1, rng=rng)
#     # Reduced steps because MP evaluates twice per step and converges faster
#     traj_bl = run_experiment_mp(model_bl, lambda x, y: grad_bilinear(x, y, B), lambda x, y: f_bilinear(x, y, B),
#                                 steps=40, problem_type="bilinear", m=1.0, M=1.0)
#     plot_advanced_diagnostics(lambda x, y: grad_bilinear(x, y, B), traj_bl, model_bl, title="Bilinear (Slingshot CP-MP)")

#     print("Running Convex-Concave CP-MP...")
#     model_cc = ConicParticlesMP(n=20, m=20, d=1, rng=rng)
#     traj_cc = run_experiment_mp(model_cc, grad_convex_concave, f_convex_concave,
#                                 steps=80, problem_type="nonlinear", L=1.0)
#     plot_advanced_diagnostics(grad_convex_concave, traj_cc, model_cc, title="Convex-Concave (Slingshot CP-MP)")

#     print("Running SC-SC CP-MP...")
#     model_sc = ConicParticlesMP(n=20, m=20, d=1, rng=rng)
#     traj_sc = run_experiment_mp(model_sc, lambda x, y: grad_sc_sc(x, y, mu=1.0), lambda x, y: f_sc_sc(x, y, mu=1.0),
#                                 steps=60, problem_type="nonlinear", L=np.sqrt(2))
#     plot_advanced_diagnostics(lambda x, y: grad_sc_sc(x, y, mu=1.0), traj_sc, model_sc, title="SC-SC (Slingshot CP-MP)")

import matplotlib.animation as animation

def run_experiment_with_history_mp(model, grad_func, f_func, steps=100, problem_type="bilinear", T=None, L=None, m=None, M=None):
    T = T or steps
    history = {
        'a': [], 'x': [],
        'b': [], 'y': [],
        'barycenter': []
    }

    for t in range(steps):
        # Record Current State
        history['a'].append(model.a.copy())
        history['x'].append(model.x.copy())
        history['b'].append(model.b.copy())
        history['y'].append(model.y.copy())

        x_bar = get_weighted_barycenter(model.a, model.x)[0]
        y_bar = get_weighted_barycenter(model.b, model.y)[0]
        history['barycenter'].append((x_bar, y_bar))

        # Get Stepsizes
        alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M)

        # Gradient helper for Mirror Prox
        def compute_system_grads(a_s, x_s, b_s, y_s):
            n, m_ = x_s.shape[0], y_s.shape[0]
            gx_total, gy_total = np.zeros_like(x_s), np.zeros_like(y_s)
            ga_total, gb_total = np.zeros(n), np.zeros(m_)
            for i in range(n):
                for j in range(m_):
                    gx, gy = grad_func(x_s[i], y_s[j])
                    val = f_func(x_s[i], y_s[j])
                    gx_total[i] += b_s[j] * float(gx)
                    gy_total[j] += a_s[i] * float(gy)
                    ga_total[i] += b_s[j] * float(val)
                    gb_total[j] += a_s[i] * float(val)
            return ga_total, gx_total, gb_total, gy_total

        # Execute Mirror Prox (Prediction-Correction) update
        model.update_mp(compute_system_grads, alpha, beta)

    return history

# ==========================================
# 2. The GIF Generator
# ==========================================
def animate_convergence(history, grad_func, filename="convergence.gif", title=""):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # --- Setup Left Plot: 2D Landscape ---
    xs, ys = np.linspace(-3, 3, 100), np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (np.sum(gx**2) + np.sum(gy**2))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.4)

    # Initialize scatter plots for the animation
    scat_p1 = axs[0].scatter([], [], c='red', edgecolors='black', zorder=5, label='P1 Particles')
    scat_p2 = axs[0].scatter([], [], c='orange', edgecolors='black', zorder=4, label='P2 Particles')
    line_bary, = axs[0].plot([], [], c='black', linestyle='-', linewidth=2, zorder=2)
    scat_bary = axs[0].scatter([], [], c='yellow', edgecolors='black', marker='*', s=300, zorder=10)

    axs[0].set_xlim(-3, 3)
    axs[0].set_ylim(-3, 3)
    axs[0].set_title(f"{title} - State Space")

    # --- Setup Middle & Right Plots: Probability Distributions ---
    axs[1].set_xlim(-3, 3)
    axs[1].set_ylim(0, 1.1)
    axs[1].set_title("P1 Strategy Mass ($a_i$ vs $x_i$)")
    axs[1].set_xlabel("Position (x)")
    axs[1].set_ylabel("Probability Mass")

    axs[2].set_xlim(-3, 3)
    axs[2].set_ylim(0, 1.1)
    axs[2].set_title("P2 Strategy Mass ($b_j$ vs $y_j$)")
    axs[2].set_xlabel("Position (y)")

    # The update function called for each frame
    def update(frame):
        # 1. Update Landscape Plot
        a, x = history['a'][frame], history['x'][frame]
        b, y = history['b'][frame], history['y'][frame]

        # Format offsets and sizes
        scat_p1.set_offsets(np.c_[x[:, 0], np.full_like(x[:, 0], -0.1)])
        scat_p1.set_sizes(np.maximum(a * 1000, 10))

        scat_p2.set_offsets(np.c_[np.full_like(y[:, 0], 0.1), y[:, 0]])
        scat_p2.set_sizes(np.maximum(b * 1000, 10))

        bary_path = np.array(history['barycenter'][:frame+1])
        if len(bary_path) > 0:
            line_bary.set_data(bary_path[:, 0], bary_path[:, 1])
            scat_bary.set_offsets(bary_path[-1])

        # 2. Update Probability Bar Charts
        axs[1].clear()
        axs[1].set_xlim(-3, 3)
        axs[1].set_ylim(0, 1.1)
        axs[1].set_title("P1 Strategy Mass ($a_i$ vs $x_i$)")
        axs[1].bar(x[:, 0], a, width=0.15, color='red', alpha=0.7, edgecolor='black')

        axs[2].clear()
        axs[2].set_xlim(-3, 3)
        axs[2].set_ylim(0, 1.1)
        axs[2].set_title("P2 Strategy Mass ($b_j$ vs $y_j$)")
        axs[2].bar(y[:, 0], b, width=0.15, color='orange', alpha=0.7, edgecolor='black')

    print(f"Generating {filename}...")
    ani = animation.FuncAnimation(fig, update, frames=len(history['a']), interval=100)
    ani.save(filename, writer='pillow')
    plt.close(fig)
    print("Done!")

if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # ==========================================
    # Case 1: Bilinear (CP-MP)
    # ==========================================
    print("Running Bilinear CP-MP Animation...")
    B_mat = np.array([[1.0]])
    model_bl = ConicParticlesMP(n=20, m=20, d=1, rng=rng)

    history_bl = run_experiment_with_history_mp(
        model_bl,
        lambda x, y: grad_bilinear(x, y, B_mat),
        lambda x, y: f_bilinear(x, y, B_mat),
        steps=50, problem_type="bilinear", m=1.0, M=1.0
    )

    animate_convergence(
        history_bl,
        lambda x, y: grad_bilinear(x, y, B_mat),
        filename="cpmp_bilinear.gif",
        title="Bilinear (Slingshot CP-MP)"
    )

    # ==========================================
    # Case 2: Convex-Concave (CP-MP)
    # ==========================================
    print("\nRunning Convex-Concave CP-MP Animation...")
    model_cc = ConicParticlesMP(n=20, m=20, d=1, rng=rng)

    history_cc = run_experiment_with_history_mp(
        model_cc,
        grad_convex_concave,
        f_convex_concave,
        steps=100, problem_type="nonlinear", L=1.0
    )

    animate_convergence(
        history_cc,
        grad_convex_concave,
        filename="cpmp_convex_concave.gif",
        title="Convex-Concave (Slingshot CP-MP)"
    )

    # ==========================================
    # Case 3: SC-SC (CP-MP)
    # ==========================================
    print("\nRunning SC-SC CP-MP Animation...")
    model_sc = ConicParticlesMP(n=20, m=20, d=1, rng=rng)

    history_sc = run_experiment_with_history_mp(
        model_sc,
        lambda x, y: grad_sc_sc(x, y, mu=1.0),
        lambda x, y: f_sc_sc(x, y, mu=1.0),
        steps=80, problem_type="nonlinear", L=np.sqrt(2)
    )

    animate_convergence(
        history_sc,
        lambda x, y: grad_sc_sc(x, y, mu=1.0),
        filename="cpmp_scsc.gif",
        title="SC-SC (Slingshot CP-MP)"
    )

    print("\nAll CP-MP animations generated successfully!")